# Basic RAG Starter

A minimal RAG baseline using minsearch. Replace the example data with your own and adapt the prompt template to your use case. Everything else should work with minimal changes.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
from minsearch import Index

client = OpenAI()

## 1. Load Your Data

Replace this example with your own data. Each document should be a dict with a few text fields you want to search over.

In [ ]:
documents = [
    {"title": "Getting Started", "content": "Install uv, create a .env file, and run uv sync to install dependencies."},
    {"title": "Running Notebooks", "content": "Use uv run jupyter lab to start Jupyter and open a notebook."},
    {"title": "Environment Variables", "content": "Keep your API keys in .env. The file is gitignored so it never ends up in your repo."},
    {"title": "Adding Dependencies", "content": "Use uv add PACKAGE_NAME to add a new dependency to pyproject.toml."},
    {"title": "Basic RAG", "content": "RAG means retrieving relevant documents from an index and passing them to the LLM as context."},
]

## 2. Build the Index

minsearch builds an in-memory TF-IDF index over the fields you specify.

In [ ]:
index = Index(
    text_fields=["title", "content"],
    keyword_fields=[],
)

index.fit(documents)

## 3. Retrieve Relevant Documents

In [ ]:
def retrieve(query, num_results=3):
    return index.search(query, num_results=num_results)

## 4. Answer the Question Using Retrieved Context

Tweak the prompt template to match what your project should actually do.

In [ ]:
PROMPT_TEMPLATE = """
Answer the user's question using only the information in the context below.
If the context doesn't contain the answer, say you don't know.

Context:
{context}

Question: {question}
""".strip()


def format_context(results):
    return "\n\n".join(f"{r['title']}\n{r['content']}" for r in results)


def answer(question):
    results = retrieve(question)
    context = format_context(results)
    prompt = PROMPT_TEMPLATE.format(context=context, question=question)

    response = client.responses.create(
        model="gpt-4o-mini",
        input=prompt,
    )
    return response.output_text

## 5. Try It Out

In [ ]:
print(answer("How do I add a new dependency?"))